# 🚀 ReparoS (2023) Training Pipeline on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hieu31/QU-solution/blob/main/notebook/train_reparos.ipynb)

Notebook này hướng dẫn toàn bộ quy trình huấn luyện mô hình **ReparoS (Base Transformer)** sửa lỗi truy vấn địa chỉ tiếng Việt trên GPU miễn phí của Google Colab:
1. **Kiểm tra GPU & Môi trường**
2. **Clone Repo & Cài đặt Thư viện**
3. **Chuẩn bị Dữ liệu Huấn luyện** (Mount Google Drive hoặc tạo data mẫu)
4. **Huấn luyện Tokenizer (SentencePiece 8K vocab)**
5. **Huấn luyện Mô hình ReparoS-Base trên GPU**
6. **Vẽ biểu đồ Loss (Train & Validation)**
7. **Thử nghiệm Inference (Top-1 & Top-k Beam Search)**
8. **Sao lưu Checkpoint & Tokenizer về Google Drive**

## 1. Kiểm tra GPU được cấp phát

In [ ]:
# Kiểm tra GPU (Chọn Runtime -> Change runtime type -> T4 GPU nếu chưa bật)
!nvidia-smi

## 2. Setup Repository & Cài đặt Dependencies

In [ ]:
import os

# Clone repository nếu chưa có
if not os.path.exists('/content/QU-solution'):
    !git clone https://github.com/Hieu31/QU-solution.git
    %cd /content/QU-solution
else:
    %cd /content/QU-solution
    !git pull

# Cài đặt package qu-solution ở chế độ editable và các dependencies cần thiết cho ReparoS
!pip install -q -e .
!pip install -q "sentencepiece>=0.2,<1" "ctranslate2>=4,<5" matplotlib

## 3. Chuẩn bị Dữ liệu Huấn luyện (Data Preparation)

Vì thư mục dữ liệu `data/` dung lượng lớn và nằm trong `.gitignore` (không có trên GitHub), bạn có thể lựa chọn:

- **Cách 1 (Khuyên dùng):** Nén thư mục `data/reparos/prepared-v1` ở máy cá nhân thành `prepared-v1.zip`, tải lên Google Drive, sau đó chạy ô bên dưới để mount và giải nén tự động.
- **Cách 2:** Tạo dữ liệu mẫu nhanh (Synthetic samples) để test toàn bộ pipeline chạy thông suốt trước khi đưa tập dữ liệu lớn vào.

In [ ]:
# --- CÁCH 1: Mount Google Drive để lấy dataset chuẩn bị từ máy cá nhân ---
from google.colab import drive
import os

drive.mount('/content/drive')

# Đường dẫn file zip trên Google Drive của bạn (thay đổi nếu cần)
drive_zip_path = '/content/drive/MyDrive/prepared-v1.zip'

if os.path.exists(drive_zip_path):
    !mkdir -p data/reparos
    !unzip -q -o {drive_zip_path} -d data/reparos/
    print("✅ Đã giải nén thành công dataset vào data/reparos/prepared-v1!")
else:
    print(f"⚠️ Không tìm thấy: {drive_zip_path}")
    print("Nếu bạn chưa tải file lên Drive, hãy chạy ô tiếp theo để tạo dữ liệu mẫu kiểm thử trước.")

In [ ]:
# --- CÁCH 2 (Dự phòng): Tạo dữ liệu mẫu nhanh nếu chưa có file zip ---
from pathlib import Path

base_dir = Path("data/reparos/prepared-v1/base")
if not (base_dir / "train.src").exists():
    print("Tạo dữ liệu mẫu nhanh để kiểm thử pipeline...")
    base_dir.mkdir(parents=True, exist_ok=True)
    
    samples = [
        ("san bay noi bai", "sân bay nội bài"),
        ("ho gom", "hồ gươm"),
        ("benh vien bach mai", "bệnh viện bạch mai"),
        ("pho hue", "phố huế"),
        ("cau giay ha noi", "cầu giấy hà nội"),
        ("nga tu so", "ngã tư sở"),
        ("cho ben thanh quan 1", "chợ bến thành quận 1"),
        ("duong nguyen hue", "đường nguyễn huệ"),
        ("vinh ha long quang ninh", "vịnh hạ long quảng ninh"),
        ("chua mot cot", "chùa một cột"),
        ("ho tay ha noi", "hồ tây hà nội"),
        ("lang chu tich ho chi minh", "lăng chủ tịch hồ chí minh"),
        ("toa nha landmark 81", "tòa nhà landmark 81"),
        ("ben xe my dinh", "bến xe mỹ đình"),
        ("ga ha noi le duan", "ga hà nội lê duẩn")
    ] * 200  # Tạo 3000 mẫu

    for split in ["train", "validation", "test"]:
        split_samples = samples if split == 'train' else samples[:300]
        with open(base_dir / f"{split}.src", "w", encoding="utf-8") as f_src, \
             open(base_dir / f"{split}.tgt", "w", encoding="utf-8") as f_tgt:
            for src, tgt in split_samples:
                f_src.write(src + "\n")
                f_tgt.write(tgt + "\n")
    print("✅ Đã tạo dữ liệu mẫu thành công tại data/reparos/prepared-v1/base!")
else:
    print("✅ Dữ liệu train/validation/test đã sẵn sàng!")

## 4. Huấn luyện Tokenizer (SentencePiece 8K vocab)

ReparoS huấn luyện mô hình Unigram SentencePiece với 8,000 subwords từ tập dữ liệu base.

In [ ]:
!reparos train-tokenizer \
  --data data/reparos/prepared-v1 \
  --output artifacts/reparos/tokenizer-v1 \
  --vocab-size 8000

## 5. Huấn luyện Mô hình ReparoS-Base Transformer trên GPU

Chạy lệnh huấn luyện với tham số chuẩn:
- `stage`: `base`
- `device`: `cuda`
- `epochs`: `10`
- `batch_size`: `128` (hoặc `64` tùy dung lượng VRAM)

In [ ]:
!reparos train \
  --stage base \
  --data data/reparos/prepared-v1 \
  --tokenizer artifacts/reparos/tokenizer-v1 \
  --output artifacts/reparos/base-v1 \
  --epochs 10 \
  --batch-size 128 \
  --device cuda

## 6. Trực quan hóa tiến trình huấn luyện (Loss Curves)

In [ ]:
import json
import os
import matplotlib.pyplot as plt

history_file = "artifacts/reparos/base-v1/training-history.jsonl"
epochs, train_losses, val_losses = [], [], []

if os.path.exists(history_file):
    with open(history_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            epochs.append(record["epoch"])
            train_losses.append(record["train_loss"])
            val_losses.append(record["validation_loss"])

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, train_losses, marker='o', linewidth=2, color='#1f77b4', label='Train Loss')
    plt.plot(epochs, val_losses, marker='s', linewidth=2, color='#d62728', label='Validation Loss')
    plt.title('ReparoS Base: Training & Validation Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Cross-Entropy Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy file training-history.jsonl. Hãy hoàn thành bước train trước!")

## 7. Thử nghiệm Inference (Dự đoán sửa lỗi câu truy vấn)

Load checkpoint tốt nhất (`best.ckpt`) và thực hiện Beam Search để sinh các gợi ý địa chỉ chuẩn hóa.

In [ ]:
from reparos.inference import ReferencePredictor

model_path = "artifacts/reparos/base-v1/best.ckpt"
tokenizer_path = "artifacts/reparos/tokenizer-v1/tokenizer.model"

# Khởi tạo predictor trên GPU (hoặc CPU nếu không có GPU)
predictor = ReferencePredictor(model_path, tokenizer_path, device="cuda")

test_queries = [
    "san bay noi bai",
    "ho gom",
    "benh vien bach mai",
    "pho hue hai ba trung",
    "cau giay ha noi",
    "cho ben thanh quan 1",
    "nga tu so"
]

print("=" * 70)
print(f"{'Truy vấn đầu vào':<30} | {'Dự đoán ReparoS (Top-1)':<35}")
print("=" * 70)

for query in test_queries:
    result = predictor.predict(query, beam_size=10)
    top1 = result["top_hypothesis"]["text"]
    print(f"{query:<30} | {top1:<35}")

print("=" * 70)

## 8. Sao lưu Checkpoint & Tokenizer vào Google Drive

Google Colab sẽ xóa sạch dữ liệu sau khi session kết thúc (thường sau vài giờ). Hãy chạy ô này để lưu trữ lâu dài mô hình đã train vào Google Drive của bạn.

In [ ]:
!mkdir -p /content/drive/MyDrive/reparos-artifacts
!cp -r artifacts/reparos /content/drive/MyDrive/reparos-artifacts/
print("🎉 Đã sao lưu thành công toàn bộ artifacts vào Google Drive (/content/drive/MyDrive/reparos-artifacts/)!")